In [148]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from geopy.distance import geodesic
import geopandas as gpd
import pulp
import folium
import osmnx as ox
import folium
from streamlit_folium import st_folium
from geopy.geocoders import Nominatim
import time
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable
from sklearn.metrics.pairwise import haversine_distances
import ssl
import certifi


In [149]:

ctx = ssl.create_default_context(cafile=certifi.where())
geolocator = Nominatim(user_agent="geoapi", ssl_context=ctx)
def reverse_geocode(lat, lon, retries=5):
    for i in range(retries):
        try:
            location = geolocator.reverse((lat, lon), exactly_one=True)
            return location.address if location else None
        except (GeocoderTimedOut, GeocoderUnavailable):
            if i < retries - 1:
                time.sleep(2)
                continue
            else:
                return None


import and cleaning

In [150]:
cam_locations = pd.read_csv(r'Automated_Traffic_Enforcement_Table.csv')

crash_data = pd.read_csv(r'Crashes_in_DC-2.csv')

/var/folders/5v/vp3xzrf13wd2c5mb7v92nnj40000gn/T/ipykernel_61778/3357910157.py:3: DtypeWarning: Columns (3,18) have mixed types. Specify dtype option on import or set low_memory=False.
  crash_data = pd.read_csv(r'Crashes_in_DC-2.csv')


In [151]:
crash_data=crash_data[crash_data['SPEEDING_INVOLVED']>=1]

In [152]:
crash_data['REPORTDATE'] = pd.to_datetime(crash_data['REPORTDATE'])
crash_data_filtered = crash_data[crash_data['REPORTDATE'].between('2010-01-01','2024-10-17',inclusive='both')]
crash_data_filtered = crash_data_filtered.rename({"LATITUDE":"latitude",'LONGITUDE':'longitude'},axis=1)

In [153]:
cam_locations['START_DATE']= pd.to_datetime(cam_locations['START_DATE'])
activecam_locations=cam_locations[cam_locations['ACTIVE_STATUS']!='Retired'].reset_index()

active_speed_cam_locations = activecam_locations[activecam_locations['ENFORCEMENT_TYPE']=='Speed']
cam_data_w_coordinates = active_speed_cam_locations[active_speed_cam_locations['CAMERA_LATITUDE'].notna()]

In [154]:
crash_data_filtered = crash_data_filtered.rename({"LATITUDE":"latitude",'LONGITUDE':'longitude'},axis=1)

crash clustering

In [155]:
def cluster_crashes(df,min_sample, eps_meters):
    
    coords = df[['latitude', 'longitude']].to_numpy()
    
    eps_km = eps_meters / 1000
    kms_per_radian = 6371.0088
    eps = eps_km / kms_per_radian

    db = DBSCAN(eps=eps, min_samples=min_sample, metric='haversine').fit(np.radians(coords))
    
    df['cluster_id'] = db.labels_
    
    return df

In [156]:
cluster_size_list = [5,10,15,20,25,30,35]
distance_list = [80,160,240,320]

In [157]:
res_dict = {}
for c in cluster_size_list:
    for d in distance_list:
        crash_clusters=cluster_crashes(crash_data_filtered,c,eps_meters=d)
        cluster_sizes = crash_clusters.groupby('cluster_id',as_index=False).size().rename({'size':"crash_count"},axis=1)
        num_clusters = cluster_sizes.shape[0]
        res_dict[f"{c}-{d}"] = num_clusters-1


In [158]:
# records = []
# for key, val in res_dict.items():
#     c, d = key.split('-')
#     records.append((int(c), float(d), val))

# res_df = pd.DataFrame(records, columns=['c', 'd', 'num_clusters_minus1'])

# # Pivot so c is index, d is column, values are num_clusters_minus1
# res_pivot = res_df.pivot(index='c', columns='d', values='num_clusters_minus1')

# res_pivot

In [159]:
# res_pivot.reset_index(drop=False)

In [160]:
#used 80m for ~standard city block size 
crash_clusters=cluster_crashes(crash_data_filtered,10,eps_meters=80)

In [161]:
#crash density per cluster
cluster_sizes = crash_clusters.groupby('cluster_id',as_index=False).size().rename({'size':"crash_count"},axis=1)

In [162]:
crash_geo_df = gpd.GeoDataFrame(crash_clusters,geometry=gpd.points_from_xy(crash_clusters['longitude'],crash_clusters['latitude']))

In [163]:
#getting central point for each cluster
cluster_centroids = crash_geo_df[crash_geo_df['cluster_id'] != -1].dissolve(by='cluster_id').centroid

crash_cluster_centroids= cluster_centroids.reset_index(drop=False)
crash_cluster_centroids.columns = ['cluster_id','crash_centroid_coords'] 


In [164]:
crash_cluster_centroids

,cluster_id,crash_centroid_coords
0,0,POINT (-76.95547 38.91697)
1,1,POINT (-77.00936 38.90724)
2,2,POINT (-76.94604 38.90078)
3,3,POINT (-76.97275 38.87436)
4,4,POINT (-76.94263 38.89255)
...,...,...
56,56,POINT (-76.94972 38.8618)
57,57,POINT (-76.99834 38.8632)
58,58,POINT (-76.95903 38.88995)
59,59,POINT (-77.0144 38.88205)


In [165]:

crash_geo_df = gpd.GeoDataFrame(
    crash_cluster_centroids,               
    geometry='crash_centroid_coords',          
    crs="EPSG:4326"                
)

In [166]:
cam_locatios_geo = gpd.GeoDataFrame(cam_data_w_coordinates,geometry=gpd.points_from_xy(cam_data_w_coordinates['CAMERA_LONGITUDE'],cam_data_w_coordinates['CAMERA_LATITUDE']),crs="EPSG:4326")[['ENFORCEMENT_SPACE_CODE','geometry']]


In [167]:
cam_locations_proj = cam_locatios_geo.to_crs(epsg=26918)
crash_locations_proj = crash_geo_df.to_crs(epsg=26918)

In [168]:
crash_locations_proj

,cluster_id,crash_centroid_coords
0,0,POINT (330467.686 4309380.266)
1,1,POINT (325771.335 4308402.72)
2,2,POINT (331246.682 4307565.772)
3,3,POINT (328867.1 4304683.49)
4,4,POINT (331523.652 4306646.326)
...,...,...
56,56,POINT (330835.615 4303247.128)
57,57,POINT (326620.169 4303493.081)
58,58,POINT (330094.628 4306388.113)
59,59,POINT (325272.306 4305615.716)


In [169]:
#getting nearest cam to crash clusters
nearest_cameras = gpd.sjoin_nearest(crash_locations_proj, cam_locations_proj, distance_col="distance_m")


In [170]:
nearest_cameras = nearest_cameras.drop_duplicates(subset='cluster_id')

In [171]:
nearest_cameras = (nearest_cameras.sort_values(by="distance_m", ascending=True).drop_duplicates(subset="cluster_id", keep="first"))

In [172]:
#merging and renaming
neartest_cam_to_crash_clusters = nearest_cameras.merge(cluster_sizes,on='cluster_id',how='left')
neartest_cam_to_crash_clusters=neartest_cam_to_crash_clusters.rename({'crash_centroid_coords':'crash_centroid_coords_projection'},axis=1)
neartest_cam_to_crash_clusters=neartest_cam_to_crash_clusters.merge(cam_locatios_geo,on='ENFORCEMENT_SPACE_CODE',how='left').rename({'geometry':"camera_coordinates"},axis=1)
neartest_cam_to_crash_clusters= neartest_cam_to_crash_clusters.merge(crash_cluster_centroids,on='cluster_id',how='left')
neartest_cam_to_crash_clusters= neartest_cam_to_crash_clusters.rename({"crash_centroid_coords":'crash_centroid_coords_coordinates','ENFORCEMENT_SPACE_CODE':"cam_name",'distance_m':'nearst_camera_distance'},axis=1).drop('index_right',axis=1)

Recomendation Model

In [173]:
neartest_cam_to_crash_clusters['camera_coordinates_longitude']=neartest_cam_to_crash_clusters['camera_coordinates'].x
neartest_cam_to_crash_clusters['camera_coordinates_latitude']=neartest_cam_to_crash_clusters['camera_coordinates'].y

neartest_cam_to_crash_clusters['crash_centroid_longitude']=neartest_cam_to_crash_clusters['crash_centroid_coords_coordinates'].x
neartest_cam_to_crash_clusters['crash_centroid_latitude']=neartest_cam_to_crash_clusters['crash_centroid_coords_coordinates'].y

In [174]:
#filtering for proximity to current camera locations
neartest_cam_to_crash_clusters_over_eighty_meters = neartest_cam_to_crash_clusters[neartest_cam_to_crash_clusters['nearst_camera_distance']>80]

In [175]:
neartest_cam_to_crash_clusters_over_eighty_meters=neartest_cam_to_crash_clusters_over_eighty_meters.set_index('cluster_id')

In [176]:
#calculating distance between crash clusters, running slow room to improve
crash_sites = neartest_cam_to_crash_clusters_over_eighty_meters[['crash_centroid_latitude', 'crash_centroid_longitude']]
crash_sites['crash_cluster_address']= crash_sites.apply(lambda x: reverse_geocode(x['crash_centroid_latitude'],x['crash_centroid_longitude']),axis=1)
# distance = pd.DataFrame([
#     [
#         geodesic((crash_sites.loc[i, 'crash_centroid_latitude'], crash_sites.loc[i, 'crash_centroid_longitude']),
#                  (crash_sites.loc[j, 'crash_centroid_latitude'], crash_sites.loc[j, 'crash_centroid_longitude'])).meters
#         for j in crash_sites.index
#     ]
#     for i in crash_sites.index
# ], columns=crash_sites.index, index=crash_sites.index)

coords_rad = np.radians(crash_sites[['crash_centroid_latitude', 'crash_centroid_longitude']].to_numpy())
distance = haversine_distances(coords_rad) * 6371000
distance = pd.DataFrame(distance, index=crash_sites.index, columns=crash_sites.index)

/var/folders/5v/vp3xzrf13wd2c5mb7v92nnj40000gn/T/ipykernel_61778/3639922425.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  crash_sites['crash_cluster_address']= crash_sites.apply(lambda x: reverse_geocode(x['crash_centroid_latitude'],x['crash_centroid_longitude']),axis=1)


In [177]:
crash_sites

,crash_centroid_latitude,crash_centroid_longitude,crash_cluster_address
cluster_id,,,
1,38.907245,-77.009361,"New York Avenue Northwest, Ward 6, Washington,..."
21,38.903851,-76.924137,"Sheriff Rd NE at Division Ave NE, Sheriff Road..."
7,38.862026,-76.997823,"Firth Sterling Avenue Southeast, Barry Farm, W..."
32,38.911473,-76.934518,"Supreme Learning Center, 1615, Kenilworth Aven..."
22,38.850804,-76.973972,"23rd Street Southeast, Buena Vista, Ward 8, Wa..."
51,38.861794,-76.959141,"Alabama Ave SE at Branch Ave SE, Alabama Avenu..."
23,38.892964,-76.950315,"Malcolm Liquors, 3845, Minnesota Avenue Northe..."
10,38.895084,-76.948880,"Shop Express, 3900, Benning Road Northeast, Wa..."
4,38.892549,-76.942625,"100, 42nd Street Northeast, Benning, Ward 7, W..."


In [178]:
# from geopy.distance import geodesic



# camera_sites = neartest_cam_to_crash_clusters[['cam_name', 'camera_coordinates_latitude', 'camera_coordinates_longitude']].drop_duplicates().reset_index(drop=True)

# distance = pd.DataFrame([
#     [
#         geodesic((neartest_cam_to_crash_clusters.loc[i, 'crash_centroid_latitude'], neartest_cam_to_crash_clusters.loc[i, 'crash_centroid_longitude']),
#                  (camera_sites.loc[j, 'camera_coordinates_latitude'], camera_sites.loc[j, 'camera_coordinates_longitude'])).meters
#         for j in range(len(camera_sites))
#     ]
#     for i in range(len(neartest_cam_to_crash_clusters))
# ], columns=camera_sites['cam_name'], index=neartest_cam_to_crash_clusters['cluster_id'])

In [179]:
# setting number of new camera needed
new_cams_needed = 20

crash_count_dict = neartest_cam_to_crash_clusters_over_eighty_meters['crash_count'].to_dict()

In [180]:


#initiate lp 
rec_model = pulp.LpProblem('cam_placement_recs',pulp.LpMinimize)

x_matrix = pulp.LpVariable.dicts('assign',(distance.index,distance.columns),0,1,cat='Binary')
y = pulp.LpVariable.dicts('open',distance.columns,0,1,cat='binary')


# equation multiplying crash density by distance from other crash cameras to make sure new camera locations arent stacked
rec_model += pulp.lpSum([(crash_count_dict[i]*2)*distance.loc[i,j]*x_matrix[i][j] for i in distance.index for j in distance.columns])



#contrains only one cam per cluster
for i in distance.index:
    rec_model += pulp.lpSum([x_matrix[i][j] for j in distance.columns])==1

for i in distance.index:
    for j in distance.columns:
        rec_model += x_matrix[i][j] <= y[j]


#contrains camera recommendations to input cameras needed
rec_model += pulp.lpSum([y[j] for j in distance.columns]) == new_cams_needed

rec_model.solve(pulp.PULP_CBC_CMD(msg=False))


1

In [181]:
#outputs
cluster_id_recomendations = [j for j in distance.columns if y[j].value() == 1]
assignments = {i: [j for j in distance.columns if x_matrix[i][j].value() == 1][0] for i in distance.index}


In [182]:
cluster_id_recomendations



[21, 7, 32, 10, 26, 18, 43, 54, 3, 25, 11, 38, 49, 9, 0, 5, 24, 42, 59, 41]

In [183]:
df = pd.DataFrame({
    'x': neartest_cam_to_crash_clusters_over_eighty_meters.loc[cluster_id_recomendations]['crash_centroid_coords_coordinates'].x,
    'y': neartest_cam_to_crash_clusters_over_eighty_meters.loc[cluster_id_recomendations]['crash_centroid_coords_coordinates'].y
})

In [184]:
df

,x,y
cluster_id,,
21,-76.924137,38.903851
7,-76.997823,38.862026
32,-76.934518,38.911473
10,-76.948880,38.895084
26,-77.022053,38.916606
18,-76.946621,38.864055
43,-77.009480,38.954984
54,-76.957159,38.884138
3,-76.972751,38.874357


Sample Viz

In [118]:
gdf = ox.geocode_to_gdf({'city': 'Washington D.C.'})

In [80]:
cam_locs_coord_only = cam_locatios_geo[['geometry']]

cam_locs_coord_only['source']='cams'

In [81]:
crash_data_viz=crash_geo_df.merge(cluster_sizes,on='cluster_id',how='left')
crash_locs_coords_only = crash_data_viz[['crash_centroid_coords','crash_count']]

crash_locs_coords_only['source']='crash'

crash_locs_coords_only = crash_locs_coords_only.rename({'crash_centroid_coords':'geometry'},axis=1)

In [82]:
master_coords = gpd.GeoDataFrame(pd.concat([cam_locs_coord_only,crash_locs_coords_only],axis=0), crs="EPSG:4326")

master_coords['radius_scaled'] = master_coords['crash_count'].fillna(1)
master_coords['radius_scaled'] = master_coords['radius_scaled']*3 * 5 

In [83]:
new_cams = neartest_cam_to_crash_clusters_over_eighty_meters.loc[cluster_id_recomendations][['crash_centroid_coords_coordinates']]

In [108]:
new_cams['crash_count'] = new_cams.index.map(crash_count_dict)

In [84]:
new_cams= new_cams.join(crash_sites['crash_cluster_address'],how='left')

In [ ]:
new_cams['source'] = 'new_cam'

new_cams = new_cams.rename({'crash_centroid_coords_coordinates':'geometry'},axis=1)

In [87]:
master_coords=pd.concat([master_coords,new_cams])

/Users/willbrennan/Desktop/Coding/school_repo/school_code/CSE 6242/myenv/lib/python3.10/site-packages/geopandas/array.py:1755: UserWarning: CRS not set for some of the concatenation inputs. Setting output's CRS as WGS 84 (the single non-null crs provided).
  return GeometryArray(data, crs=_get_common_crs(to_concat))


In [70]:
maps = gdf.explore(name="DC", height=600, width=800)

In [89]:
#red = camera

maps = master_coords.explore(
    m=maps,
    column='source',
    cmap=['red','blue','black'],
    legend=True,
    marker_type='circle',
    radius='radius_scaled',  
    tooltips=['crash_count']
)

In [90]:
folium.LayerControl().add_to(maps)

maps.save("map.html")

- get closest camera location to each crash grouping
- leaflet uses geojson, structure output accordingly

